# PHASE 3B - PARALLEL PART 3/4: SEEDS 92 TO 116
Evaluates 25 placebo vectors (Seeds 92 to 116) under Layer 8 Linear Decay ($K=16, \alpha_0=18.0$) autoregressive generation with BERTScore reference-preference.

In [1]:
!pip install -q evaluate bert_score bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.1 MB/s eta 0:00:00


In [2]:
import os, json, time, math, torch, numpy as np, pandas as pd, sys
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print('PyTorch Version:', torch.__version__, flush=True)
print('CUDA Available:', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('Device Count:', torch.cuda.device_count(), flush=True)
    for d in range(torch.cuda.device_count()):
        print(f'  GPU {d}:', torch.cuda.get_device_name(d), flush=True)

PyTorch Version: 2.10.0+cu128
CUDA Available: True
Device Count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
# Load Qwen2.5-7B-Instruct Model and BERTScore Metric
model_id = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)

model.eval()
bertscore = evaluate.load('bertscore')
print('✅ Model and BERTScore successfully loaded!', flush=True)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model and BERTScore successfully loaded!


In [4]:
# Define Layer 8 Linear Decay Hook (alpha_0=18.0, K=16) & Load Dataset
possible_paths = [
    './data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json'
]
data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

test_data = full_dataset[-500:]
eval_subset = test_data[:100]

def make_device_safe_hook(v_vector, alpha_0=18.0, K=16):
    step_counter = 0
    def hook_fn(module, input_tensor, output_tensor):
        nonlocal step_counter
        step_counter += 1
        if 1 <= step_counter <= K:
            alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K)
            if isinstance(output_tensor, tuple):
                cur_tensor = output_tensor[0]
                v_curr = v_vector.to(device=cur_tensor.device, dtype=cur_tensor.dtype)
                modified = cur_tensor + alpha_t * v_curr
                return (modified,) + output_tensor[1:]
            else:
                v_curr = v_vector.to(device=output_tensor.device, dtype=output_tensor.dtype)
                return output_tensor + alpha_t * v_curr
        return output_tensor
    return hook_fn

target_layer_module = model.model.layers[8]
hidden_dim = model.config.hidden_size
print(f'✅ Loaded {len(test_data)} test items (evaluating N={len(eval_subset)} per seed) on Layer 8!', flush=True)

✅ Loaded 500 test items (evaluating N=100 per seed) on Layer 8!


In [5]:
# Run Part 3 Evaluation (Seeds 92 to 116)
csv_filename = 'placebo_part3.csv'
completed_seeds = set()

if os.path.exists(csv_filename):
    df_existing = pd.read_csv(csv_filename)
    if 'seed' in df_existing.columns:
        completed_seeds = set(df_existing['seed'].tolist())
    print(f'🔄 Resuming! Found {len(completed_seeds)} completed seeds in {csv_filename}.', flush=True)
else:
    df_init = pd.DataFrame(columns=['seed', 'accuracy'])
    df_init.to_csv(csv_filename, index=False)
    print(f'🆕 Initialized {csv_filename}.', flush=True)

print('========================================================================', flush=True)
print('🚀 RUNNING PART 3/4 BENCHMARK (SEEDS 92 TO 116):', flush=True)
print('========================================================================', flush=True)

for seed in range(92, 117):
    if seed in completed_seeds:
        print(f'  [Part 3 | Seed {seed:03d}] -> ALREADY COMPLETED.', flush=True)
        continue
    torch.manual_seed(seed)
    v_rand_raw = torch.randn(hidden_dim, dtype=torch.float32)
    v_rand = v_rand_raw / v_rand_raw.norm(p=2)
    rand_gen, rand_refs, rand_hals = [], [], []
    for item in eval_subset:
        q_text = item['question']
        prompt = f'<|im_start|>user\n{q_text}<|im_end|>\n<|im_start|>assistant\n'
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        hook_h = target_layer_module.register_forward_hook(make_device_safe_hook(v_rand, alpha_0=18.0, K=16))
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        hook_h.remove()
        gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
        rand_gen.append(gen_text)
        rand_refs.append(item.get('right_answer', item.get('positive_answer')))
        rand_hals.append(item['hallucinated_answer'])
    r_bs_ref = bertscore.compute(predictions=rand_gen, references=rand_refs, model_type='bert-base-multilingual-cased')['f1']
    r_bs_hal = bertscore.compute(predictions=rand_gen, references=rand_hals, model_type='bert-base-multilingual-cased')['f1']
    r_acc = sum(1 for r, h in zip(r_bs_ref, r_bs_hal) if r > h) / len(eval_subset) * 100.0
    df_new = pd.DataFrame([{'seed': seed, 'accuracy': r_acc}])
    df_new.to_csv(csv_filename, mode='a', header=False, index=False)
    completed_seeds.add(seed)
    print(f'  [Part 3 | Seed {seed:03d}] -> Acc: {r_acc:.2f}% -> SAVED!', flush=True)

df_res = pd.read_csv(csv_filename)
print('========================================================================', flush=True)
print('📊 PART 3 COMPLETE (placebo_part3.csv)!', flush=True)
print(f'   Mean Accuracy: {df_res["accuracy"].mean():.2f}% ± {df_res["accuracy"].std():.2f}%', flush=True)
print('========================================================================', flush=True)

🆕 Initialized placebo_part3.csv.
🚀 RUNNING PART 3/4 BENCHMARK (SEEDS 92 TO 116):


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Part 3 | Seed 092] -> Acc: 60.00% -> SAVED!
  [Part 3 | Seed 093] -> Acc: 68.00% -> SAVED!
  [Part 3 | Seed 094] -> Acc: 57.00% -> SAVED!
  [Part 3 | Seed 095] -> Acc: 54.00% -> SAVED!
  [Part 3 | Seed 096] -> Acc: 57.00% -> SAVED!
  [Part 3 | Seed 097] -> Acc: 62.00% -> SAVED!
  [Part 3 | Seed 098] -> Acc: 52.00% -> SAVED!
  [Part 3 | Seed 099] -> Acc: 60.00% -> SAVED!
  [Part 3 | Seed 100] -> Acc: 62.00% -> SAVED!
  [Part 3 | Seed 101] -> Acc: 61.00% -> SAVED!
  [Part 3 | Seed 102] -> Acc: 59.00% -> SAVED!
  [Part 3 | Seed 103] -> Acc: 60.00% -> SAVED!
  [Part 3 | Seed 104] -> Acc: 52.00% -> SAVED!
  [Part 3 | Seed 105] -> Acc: 60.00% -> SAVED!
  [Part 3 | Seed 106] -> Acc: 57.00% -> SAVED!
  [Part 3 | Seed 107] -> Acc: 58.00% -> SAVED!
  [Part 3 | Seed 108] -> Acc: 59.00% -> SAVED!
  [Part 3 | Seed 109] -> Acc: 52.00% -> SAVED!
  [Part 3 | Seed 110] -> Acc: 59.00% -> SAVED!
  [Part 3 | Seed 111] -> Acc: 52.00% -> SAVED!
  [Part 3 | Seed 112] -> Acc: 53.00% -> SAVED!
  [Part 3 | S